# Lewisham · `decision` (web: "Decision")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · All `decision` values

A `terms` aggregation on `decision.raw` (exact values, case-sensitive) plus a `missing`
aggregation for records with no decision. `size: 0` = no records, only the counts.
The `shown_as` column uses `repr()` so hidden characters (tabs, spaces) are visible.

In [2]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": lewisham_filter(),
    "aggs": {
        "decision": {"terms": {"field": "decision.raw", "size": 100}},
        "no_decision": {"missing": {"field": "decision.raw"}},
    },
}
data = search(body)

total = data["hits"]["total"]["value"]
buckets = data["aggregations"]["decision"]["buckets"]
decision_counts = pd.DataFrame(
    [(b["key"], b["doc_count"]) for b in buckets] + [(None, data["aggregations"]["no_decision"]["doc_count"])],
    columns=["decision", "count"],
).sort_values("count", ascending=False, ignore_index=True)

print("Lewisham applications:", total, "| sum of counts:", decision_counts["count"].sum())

Lewisham applications: 29021 | sum of counts: 29021


In [3]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "Decision" (extracted 2026-11-09).
SCHEMA_DECISION = [
    "Approved", "Called in by Secretary of State", "Closed", "Comment Issued",
    "Declined to Determine", "Not Required", "Refused", "Split Decision", "Withdrawn",
    "No Objection to Proposal (OBS only)", "Objection Raised to Proposal (OBS only)",
    "Opinion Issued", "Pre-application Advice Case Completed", "Referred to Mayor",
]

decision_counts["shown_as"] = decision_counts["decision"].map(repr)
decision_counts["in_schema"] = decision_counts["decision"].isin(SCHEMA_DECISION)
decision_counts["share_%"] = (100 * decision_counts["count"] / total).round(2)
decision_counts

,decision,count,shown_as,in_schema,share_%
0,Approved,20735,'Approved',True,71.45
1,Refused,4425,'Refused',True,15.25
2,NaN,1760,nan,False,6.06
3,Withdrawn,939,'Withdrawn',True,3.24
4,Not Required,730,'Not Required',True,2.52
5,No Objection to Proposal (OBS only),223,'No Objection to Proposal (OBS only)',True,0.77
6,Closed,113,'Closed',True,0.39
7,Comment Issued,66,'Comment Issued',True,0.23
8,Objection Raised to Proposal (OBS only),29,'Objection Raised to Proposal (OBS only)',True,0.10
9,Not required,1,'Not required',False,0.00


### Findings · Step 1

Live API, 2026-11-11, 29,021 Lewisham records (before cleaning):

| `decision` | Count | Share | In schema |
|---|---|---|---|
| Approved | 20,735 | 71.45% | yes |
| Refused | 4,425 | 15.25% | yes |
| (none) | 1,760 | 6.06% | — |
| Withdrawn | 939 | 3.24% | yes |
| Not Required | 730 | 2.52% | yes |
| No Objection to Proposal (OBS only) | 223 | 0.77% | yes |
| Closed | 113 | 0.39% | yes |
| Comment Issued | 66 | 0.23% | yes |
| Objection Raised to Proposal (OBS only) | 29 | 0.10% | yes |
| Not required | 1 | 0.00% | no |

- **One case variant:** `Not required` (1) for the schema's `Not Required`.
- **No hidden characters:** `shown_as` shows no tabs or spaces (Brent had 3 `"\tApproved"`).
- **No Split Decision as a decision:** status has 11 Split Decision; as in Brent, it
  looks like an appeal outcome. Not checked yet.
- **1,760 have no decision:** 1,161 of them are records with no status too (Step 2 of
  the status notebook; 1,151 are dropped as empty). The rest is not broken down yet.

## Step 2 · The `Not required` record

The one lower-case variant of the schema value `Not Required` (Step 1). Fetch it with the
fields that matter for the outcome, to check it is a plain case variant.

In [4]:
FIELDS = ["id", "lpa_app_no", "application_type", "application_type_full", "status", "decision",
          "decision_date", "valid_date", "bo_system", "last_updated", "url_planning_app", "description"]
body = {"size": 5, "_source": FIELDS,
        "query": {"bool": {"filter": [lewisham_filter(), {"term": {"decision.raw": "Not required"}}]}}}
hits = search(body)["hits"]["hits"]
print(len(hits), "record(s)")
for h in hits:
    for k in FIELDS:
        print(f"{k}: {h['_source'].get(k)}")

1 record(s)
id: Lewisham-DC_15_09720
lpa_app_no: DC/15/09720
application_type: All Other
application_type_full: Full planning permission
status: Not Required
decision: Not required
decision_date: 22/02/2018
valid_date: None
bo_system: FME Legacy Data Import
last_updated: 2025-08-14T13:21:02.327+01:00
url_planning_app: None
description: The demolition of the existing garages on Site A, Land on Western Side of Millbank Way, Lee Green Estate, Cambridge Drive SE12, the construction of a part two/part three storey building to provide 4 one bedroom, 4 two bedroom and 4 three bedroom self-contained flats together with associated landscaping, amenity space, refuse storage and detached cycle store. (Amended Description)


### Findings · Step 2

One record, Lewisham-DC_15_09720 (DC/15/09720):

- **A plain case variant:** its status is `Not Required` (schema spelling); only
  `decision` has the lower-case `r`. Mapping it to `Not Required` makes the two agree.
- **A legacy import** (`bo_system` FME Legacy Data Import): no valid_date and no URL, the
  same pattern as Brent's legacy records. Not checked for Lewisham as a whole yet.
- **Odd for "Not Required":** application_type_full is "Full planning permission" (12 flats
  on garage land, Lee Green Estate, decided 22/02/2018). Noted only; not a cleaning
  question.

## Step 3 · Records with a status but no decision

Step 1 found 1,760 records with no decision; 1,161 of them have no status either (status
notebook, Step 2). Here: the rest, split by `status`, with how many have a decision_date.

In [5]:
body = {
    "size": 0,
    "track_total_hits": True,
    "query": {"bool": {"filter": [lewisham_filter(), {"exists": {"field": "status.raw"}}],
                       "must_not": [{"exists": {"field": "decision.raw"}}]}},
    "aggs": {"status": {"terms": {"field": "status.raw", "size": 100},
                        "aggs": {"no_decision_date": {"missing": {"field": "decision_date"}}}}},
}
data = search(body)
print("Status but no decision:", data["hits"]["total"]["value"])
rows = [(b["key"], b["doc_count"], b["no_decision_date"]["doc_count"]) for b in data["aggregations"]["status"]["buckets"]]
by_status = pd.DataFrame(rows, columns=["status", "count", "no_decision_date"])
by_status["with_decision_date"] = by_status["count"] - by_status["no_decision_date"]
by_status

Status but no decision: 599


,status,count,no_decision_date,with_decision_date
0,Application Under Consideration,553,553,0
1,Superseded,23,0,23
2,Lapsed,15,0,15
3,Application Invalid,8,8,0


### Findings · Step 3

599 records have a status but no decision:

| status | Count | With decision_date |
|---|---|---|
| Application Under Consideration | 553 | 0 |
| Superseded | 23 | 23 |
| Lapsed | 15 | 15 |
| Application Invalid | 8 | 0 |

- **561 are as expected:** the 553 pending applications and 8 invalid ones have no
  decision and no decision_date.
- **38 have lost their decision:** every Superseded (23) and Lapsed (15) record here has
  a decision_date but no decision. Something was decided, but the Datahub doesn't say
  what. In Brent, status Lapsed did not show the decision either (35 Lapsed were
  Refused), so it cannot be read from the status.

## Step 4 · The 38 Superseded / Lapsed records with no decision

Listed with their dates, source system and whether they have a portal URL.

In [6]:
FIELDS = ["id", "status", "application_type", "valid_date", "decision_date", "bo_system",
          "url_planning_app", "last_updated", "description"]
body = {"size": 100, "_source": FIELDS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"status.raw": ["Superseded", "Lapsed"]}}],
                           "must_not": [{"exists": {"field": "decision.raw"}}]}}}
lost = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
lost["url"] = lost.pop("url_planning_app").notna()
lost["last_updated"] = lost["last_updated"].str.slice(0, 10)
lost["description"] = lost["description"].str.slice(0, 50)
print(len(lost), "records")
print(lost[["status", "bo_system", "url"]].value_counts(dropna=False).to_string())
print()
print(lost.sort_values(["status", "id"]).to_string(index=False))

38 records
status      bo_system               url  
Superseded  FME Legacy Data Import  False    23
Lapsed      FME Legacy Data Import  False    15

                   id     status application_type valid_date decision_date              bo_system last_updated                                        description   url
   Lewisham-DC0251010     Lapsed        All Other       None    20/05/2003 FME Legacy Data Import   2025-03-21                     Construction of a 3 bed house. False
  Lewisham-DC06_62027     Lapsed        All Other       None    13/03/2007 FME Legacy Data Import   2025-03-21 The demolition of the Maypole Inn, 1 Mantle Road S False
  Lewisham-DC07_67309     Lapsed        All Other       None    22/05/2008 FME Legacy Data Import   2025-03-21 The alteration and conversion of the lower ground  False
  Lewisham-DC08_68294     Lapsed        All Other       None    24/11/2008 FME Legacy Data Import   2025-03-21 The construction of a roof extension in the rear r False
  Lewisham

### Findings · Step 4

- **All 38 are legacy imports** (`bo_system` FME Legacy Data Import), with no valid_date
  and no portal URL. Decision dates run from 1987 to 2018.
- **Updated together:** 37 have last_updated 2025-03-20/21 (one bulk update); 1 has
  2025-12-02.
- **No portal link,** so a person cannot open them from the record; a portal search by
  reference would be needed.
- **Many are outline or larger schemes** ("Outline for…", "Erection of 46 houses"):
  Superseded and Lapsed fit outline permissions that were replaced or not used, but that
  is not confirmed.
- One description is a placeholder: "No description provided by LDMS" (DC27747).
- Their ids have older formats (DC27253, DC0251010, DCDE151ATP), unlike the DC/yy/nnnnnn
  of the CSV import.

## Step 5 · `decision` × `status`

A `terms` aggregation on decision with a `status` sub-aggregation (`missing` shown as
"(none)"), before cleaning. Shows where the two fields disagree.

In [7]:
body = {
    "size": 0,
    "query": lewisham_filter(),
    "aggs": {"decision": {"terms": {"field": "decision.raw", "size": 100, "missing": "(none)"},
                          "aggs": {"status": {"terms": {"field": "status.raw", "size": 100, "missing": "(none)"}}}}},
}
buckets = search(body)["aggregations"]["decision"]["buckets"]
pairs = pd.DataFrame([(d["key"], s["key"], s["doc_count"]) for d in buckets for s in d["status"]["buckets"]],
                     columns=["decision", "status", "count"])
crosstab = pairs.pivot_table(index="status", columns="decision", values="count", aggfunc="sum",
                             fill_value=0, margins=True, margins_name="total")
crosstab.sort_values("total", ascending=False)

decision,(none),Approved,Closed,Comment Issued,No Objection to Proposal (OBS only),Not Required,Not required,Objection Raised to Proposal (OBS only),Refused,Withdrawn,total
status,,,,,,,,,,,
total,1760,20735,113,66,223,730,1,29,4425,939,29021
Approved,0,15062,0,0,0,0,0,0,0,0,15062
Completed,0,3608,0,0,0,13,0,0,78,0,3699
Refused,0,0,0,0,0,0,0,0,3689,0,3689
Lapsed,15,1445,0,0,0,0,0,0,15,0,1475
(none),1161,1,2,12,0,0,0,2,0,5,1183
Withdrawn,0,3,0,0,0,0,0,0,13,934,950
Not Required,0,0,0,0,0,715,1,0,0,0,716
Application Under Consideration,553,0,0,0,0,0,0,0,0,0,553


### Findings · Step 5

**Mostly the same:** 20,803 records (72%) have status equal to decision, counting the
`Not required` case variant (Approved 15,062, Refused 3,689, Withdrawn 934, Not Required
717, No Objection 222, Closed 111, Comment Issued 44, Objection Raised 24).

**Appeal outcomes in status, as in Brent:** the council's decision stays in `decision`.

| status | decision Refused | decision Approved | other |
|---|---|---|---|
| Dismissed (446) | 443 | 3 | — |
| Allowed (140) | 138 | 2 | — |
| Appeal Received (30) | 30 | — | — |
| Split Decision (11) | 11 | — | — |
| Appeal in Progress (1) | 1 | — | — |

5 appeals follow an approval; why is not checked.

**The status moved on after the decision:**
- Completed: 3,608 Approved, 78 Refused, 13 Not Required.
- Lapsed: 1,445 Approved, 15 Refused, 15 no decision (Step 4). As in Brent, Lapsed does
  not mean a permission existed.
- Commenced: 136 Approved, 2 Refused, 1 Not Required.
- Superseded: 473 Approved, 1 Not Required, 23 no decision (Step 4).

**Disagreements, not explained yet:**
- status Withdrawn with decision Refused (13) or Approved (3).
- status Closed (129): Comment Issued 10, Objection Raised 3, Refused 3, Approved 1, No
  Objection 1 besides 111 Closed.
- status Application Invalid with decision Approved (1) or Refused (1).

**No pending application has a decision:** all 553 Application Under Consideration have
none. Brent had 2 (decision_not_issued); Lewisham has no such case.

## Step 6 · Status Withdrawn with decision Refused or Approved

The 16 records from Step 5, with dates, appeal fields and source system.

In [8]:
FIELDS = ["id", "decision", "decision_date", "valid_date", "appeal_decision", "appeal_decision_date",
          "application_type", "bo_system", "url_planning_app", "last_updated", "description"]
body = {"size": 100, "_source": FIELDS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter(), {"term": {"status.raw": "Withdrawn"}},
                                      {"terms": {"decision.raw": ["Refused", "Approved"]}}]}}}
withdrawn = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
withdrawn["url"] = withdrawn.pop("url_planning_app").notna()
withdrawn["last_updated"] = withdrawn["last_updated"].str.slice(0, 10)
withdrawn["description"] = withdrawn["description"].str.slice(0, 45)
print(len(withdrawn), "records")
print(withdrawn[["decision", "bo_system", "url"]].value_counts(dropna=False).to_string())
print()
print(withdrawn.sort_values(["decision", "id"]).to_string(index=False))

16 records
decision  bo_system        url 
Refused   CSV Data Import  True    13
Approved  CSV Data Import  True     3

                   id decision decision_date valid_date appeal_decision appeal_decision_date application_type       bo_system last_updated                                   description  url
Lewisham-DC_22_125824 Approved    03/05/2022 07/03/2022       Withdrawn           07/12/2022        All Other CSV Data Import   2026-11-10 Advertisement Consent in respect of the insta True
Lewisham-DC_25_138662 Approved    15/04/2025 16/01/2025       Withdrawn           16/04/2025      Householder CSV Data Import   2026-11-10 Construction of a rear roof extension with do True
Lewisham-DC_25_140617 Approved    13/02/2026 30/06/2025       Withdrawn           16/07/2026        All Other CSV Data Import   2026-11-10 Partial retrospective planning permission for True
Lewisham-DC_19_113464  Refused    21/01/2020 26/11/2019       Withdrawn           16/10/2020      Householder CSV Data I

### Findings · Step 6

- **All 16 are withdrawn appeals, not withdrawn applications:** every one has
  appeal_decision Withdrawn and an appeal_decision_date after the decision_date. The
  council's decision (13 Refused, 3 Approved) stands; status shows the appeal outcome, as
  for Allowed / Dismissed (Step 5).
- **Different from Brent:** 11 of Brent's 43 withdrawn appeals have status Refused; in
  Lewisham these 16 have status Withdrawn. So status Withdrawn in Lewisham can mean
  the appeal was withdrawn.
- All are CSV Data Import with a URL, decided 2020–2026; all last_updated 2026-11-10.
- 3 appeals follow an approval; why (e.g. against conditions) is not checked.

## Step 7 · The 129 records with status Closed

Broken down by decision, `lpa_app_no` prefix, source system and appeal fields; then the 18
whose decision is not Closed are listed.

In [9]:
FIELDS = ["id", "lpa_app_no", "decision", "decision_date", "valid_date", "appeal_decision",
          "application_type", "bo_system", "url_planning_app", "description"]
body = {"size": 500, "_source": FIELDS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter(), {"term": {"status.raw": "Closed"}}]}}}
closed = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
closed["prefix"] = closed["lpa_app_no"].str.extract(r"^([A-Z]+)", expand=False)
closed["url"] = closed.pop("url_planning_app").notna()
closed["has_decision_date"] = closed["decision_date"].notna()
print(len(closed), "records")
print(closed[["decision", "prefix", "bo_system", "appeal_decision", "has_decision_date"]]
      .value_counts(dropna=False).to_string())

129 records
decision                                 prefix  bo_system               appeal_decision  has_decision_date
Closed                                   DC      CSV Data Import         NaN              True                 102
Comment Issued                           PRE     CSV Data Import         NaN              True                  10
Closed                                   DC      FME Legacy Data Import  NaN              True                   8
Refused                                  DC      CSV Data Import         NaN              True                   3
Objection Raised to Proposal (OBS only)  PRE     CSV Data Import         NaN              True                   3
Approved                                 DC      CSV Data Import         NaN              True                   1
Closed                                   PRE     CSV Data Import         NaN              True                   1
No Objection to Proposal (OBS only)      PRE     CSV Data Import         Na

In [10]:
other = closed[closed["decision"] != "Closed"].copy()
other["description"] = other["description"].str.slice(0, 60)
print(len(other), "records")
print(other.drop(columns=["prefix", "has_decision_date"]).sort_values(["decision", "id"]).to_string(index=False))

18 records
                    id    lpa_app_no                                decision decision_date valid_date appeal_decision application_type       bo_system                                                  description  url
 Lewisham-DC_20_119156  DC/20/119156                                Approved    20/04/2022 11/11/2020            None        All Other CSV Data Import Phased outline planning application for demolition of the bu True
Lewisham-PRE_19_110816 PRE/19/110816                          Comment Issued    22/03/2019 13/02/2019            None        All Other CSV Data Import The conversion of two self-contained flats into a single dwe True
Lewisham-PRE_19_113483 PRE/19/113483                          Comment Issued    25/09/2019 29/08/2019            None        All Other CSV Data Import Concept Meeting: Demolition of two garages and erection of a True
Lewisham-PRE_19_114956 PRE/19/114956                          Comment Issued    14/02/2020 27/01/2020            None    

### Findings · Step 7

129 records with status Closed, none with an appeal decision, all with a decision_date:

| decision | prefix | Count |
|---|---|---|
| Closed | DC/ (CSV 102, legacy 8) | 110 |
| Closed | PRE/ | 1 |
| Comment Issued / Objection Raised / No Objection | PRE/ | 14 |
| Refused | DC/ | 3 |
| Approved | DC/ | 1 |

- **111 agree** (status and decision Closed).
- **14 PRE/ records:** advice given (Comment Issued 10, Objection Raised 3, No Objection 1),
  then the case was closed. The decision is the advice; status Closed is the stage after.
  Consistent, not an error.
- **4 DC/ records with a council decision** (Refused 3, Approved 1; 2019–2022, no appeal):
  status Closed after a decision. Why is not explained; the decision is the outcome to
  report.
- **PRE/ cases are in the Lewisham data** (here and in the status notebook, Step 3).
  Whether they count as planning applications is a separate question; not checked how
  many there are.

## Step 8 · Records by `lpa_app_no` prefix, with decision values

`lpa_app_no` has no `.raw` keyword field, so it cannot be aggregated: every Lewisham record
is fetched (`search_after`, sorted by `id`) and counted in pandas. The prefix is the
leading letters of `lpa_app_no` (e.g. `DC/21/120320` → `DC`, `DC27253` → `DC`).

In [11]:
def fetch_all(query: dict, fields: list[str]) -> pd.DataFrame:
    rows, after = [], None
    while True:
        body = {"size": 5000, "query": query, "_source": fields, "sort": [{"id": "asc"}]}
        if after:
            body["search_after"] = after
        hits = search(body)["hits"]["hits"]
        if not hits:
            return pd.DataFrame(rows, columns=fields)
        rows += [h["_source"] for h in hits]
        after = hits[-1]["sort"]


everything = fetch_all(lewisham_filter(), ["id", "lpa_app_no", "status", "decision", "bo_system"])
everything["prefix"] = everything["lpa_app_no"].str.extract(r"^([A-Za-z]*)", expand=False).replace("", "(digits)")
print("Records:", len(everything), "| no lpa_app_no:", everything["lpa_app_no"].isna().sum())
print()
print(everything["prefix"].value_counts(dropna=False).to_string())

Records: 29021 | no lpa_app_no: 0

prefix
DC          28861
(digits)      106
PRE            41
TEST            6
DCLE            4
DCDE            3


In [12]:
pd.crosstab(everything["decision"].fillna("(none)"), everything["prefix"].fillna("(none)"), margins=True, margins_name="total")

prefix,(digits),DC,DCDE,DCLE,PRE,TEST,total
decision,,,,,,,
(none),35,1714,2,2,5,2,1760
Approved,54,20679,0,0,0,2,20735
Closed,0,110,0,0,3,0,113
Comment Issued,0,43,0,0,23,0,66
No Objection to Proposal (OBS only),0,220,0,0,3,0,223
Not Required,1,728,0,0,0,1,730
Not required,0,1,0,0,0,0,1
Objection Raised to Proposal (OBS only),0,23,0,0,6,0,29
Refused,15,4406,1,2,0,1,4425


### Findings · Step 8

29,021 records, every one with an `lpa_app_no`:

| prefix | Count |
|---|---|
| DC | 28,861 |
| (digits only, e.g. 95/038596) | 106 |
| PRE | 41 |
| TEST | 6 |
| DCLE | 4 |
| DCDE | 3 |

- **Only 41 PRE/ records** (0.14%). Their decisions: Comment Issued 23, Objection Raised 6,
  Closed 3, No Objection 3, Withdrawn 1, none 5. No Approved or Refused.
- **The opinion-type values are not PRE/ only:** most are DC/ records: No Objection to
  Proposal (OBS only) 220 of 223, Objection Raised 23 of 29, Comment Issued 43 of 66. So they
  cannot be used to find PRE/ cases, and "OBS" is not tied to PRE/.
- **6 TEST records:** Brent had test records (55/), dropped. These need a look.
- 106 references are digits only (older format); 7 are DCLE / DCDE (legacy formats, Step 4).

## Step 9 · The 6 `TEST` records

Fetched by id with the fields that show whether they are real applications (Brent's test
records said "*** TEST APPLICATION ***" or "Test location").

In [13]:
FIELDS = ["id", "lpa_app_no", "status", "decision", "decision_date", "valid_date", "application_type",
          "bo_system", "last_updated", "site_name", "street_name", "postcode", "url_planning_app", "description"]
test_ids = everything.loc[everything["prefix"] == "TEST", "id"].tolist()
body = {"size": len(test_ids), "_source": FIELDS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": test_ids}}]}}}
tests = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
tests["last_updated"] = tests["last_updated"].str.slice(0, 10)
tests["url_planning_app"] = tests["url_planning_app"].notna()
tests.T

,0,1,2,3,4,5
id,Lewisham-TEST_13_022223_X,Lewisham-TEST_18_022222,Lewisham-TEST_22_022222,Lewisham-TEST_22_022223,Lewisham-TEST_69_022223,Lewisham-TEST_80_022222_X
lpa_app_no,TEST/13/022223/X,TEST/18/022222,TEST/22/022222,TEST/22/022223,TEST/69/022223,TEST/80/022222/X
status,Application Under Consideration,Application Invalid,Approved,Not Required,Application Invalid,Application Invalid
decision,NaN,Refused,Approved,Not Required,Approved,NaN
decision_date,NaN,NaN,06/03/2024,18/09/2023,NaN,NaN
valid_date,23/09/2026,22/09/2026,04/08/2026,18/09/2023,28/09/2026,22/09/2026
application_type,Householder,All Other,All Other,Householder,All Other,Householder
bo_system,CSV Data Import,CSV Data Import,CSV Data Import,CSV Data Import,CSV Data Import,CSV Data Import
last_updated,2026-11-10,2026-11-10,2026-10-06,2026-06-22,2026-11-10,2026-11-10
site_name,"FLAT 2, 7 HECTOR CRESCENT, LONDON, SE6","LEE GREEN TELEPHONE EXCHANGE, LEE HIGH ROAD, L...","68 MANOR LANE, LONDON, SE13 5QP","115A HITHER GREEN LANE, LONDON, SE13 6QF","FLAT 10, BROADSIDE HOUSE, CREEKSIDE, LONDON, S...","MCDONALDS, THE GARDEN GATE, OLD BROMLEY ROAD, ..."


In [14]:
# The descriptions in full (first 200 characters), since they show what the records are.
for _, row in tests.iterrows():
    print(row["id"], "|", row["description"][:200], end="\n\n")

Lewisham-TEST_13_022223_X | TEST  DESCRIPTIONTEST  DESCRIPTIONTEST  DESCRIPTION

Lewisham-TEST_18_022222 | Application for a Lawful Development Certificate [(existing/proposed)] submitted under Section [191/2] of the Town & Country Planning Act 1990 (as amended) [insert relevant legislation] and Class [ins

Lewisham-TEST_22_022222 | Construction of a single  storey extension to the rear of 133a Deptford High Street, SE8.A Proposed Test Development  - delibilate seppelling mistakio Carstairs One New build House and a shed and gara

Lewisham-TEST_22_022223 | Test Test

Lewisham-TEST_69_022223 | Test Case to illustrate the Golden RulesTEST CASE NOT A LIVE CASE

Lewisham-TEST_80_022222_X | This is test data to produce a draft Decison Notice for Full Planning application test TEST



### Findings · Step 9

**All 6 are test records:**

| id | status / decision | description says |
|---|---|---|
| TEST/13/022223/X | Application Under Consideration / — | "TEST DESCRIPTION" ×3 |
| TEST/18/022222 | Application Invalid / Refused | a template with "[insert relevant legislation]" |
| TEST/22/022222 | Approved / Approved | "A Proposed Test Development - delibilate seppelling mistakio…" |
| TEST/22/022223 | Not Required / Not Required | "Test Test" |
| TEST/69/022223 | Application Invalid / Approved | "TEST CASE NOT A LIVE CASE" |
| TEST/80/022222/X | Application Invalid / — | "This is test data to produce a draft Decison Notice" |

- **They look like real records:** real Lewisham addresses (68 Manor Lane SE13, 115A Hither
  Green Lane SE13, …), portal URLs, CSV Data Import, valid dates in 2023–2026.
- **They would distort answers:** one would count as pending, one as approved (2024), one
  as Not Required.
- **Like Brent's 55/ records:** Brent's rule `drop_test_records` lists them by id, and a
  new `55/` reference stops the load. Here the marker is the `TEST` prefix.
- **Recently touched:** 4 of 6 last_updated 2026-11-10, and their valid_dates are in
  Sept 2026, so new test records may appear.

## Step 10 · Type check

Aggregations show every key as text, whatever its JSON type. So the Python type of each
`decision` value is checked on the records fetched in Step 8 (same method as Brent's).

In [15]:
types = everything["decision"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Lewisham records:", len(everything))
print(types.value_counts().to_string())
not_str = everything[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))

Lewisham records: 29021
decision
str         27261
NoneType     1760
values that are not strings: 0


### Findings · Step 10

- **Every filled `decision` is a string** (27,261); 1,760 are missing, as in Step 1. No
  numbers or other types, so `decision` matches the schema's text type.

## Step 11 · `decision` after cleaning

As in the status notebook (Step 5): the drops and the status and decision rules of
`semantic/lewisham/cleaning.yaml` applied by hand, reading ids and values from the file.
Then the counts that `decision.yaml` gives.

In [16]:
import yaml

rules = yaml.safe_load((PROJECT_ROOT / "semantic/lewisham/cleaning.yaml").read_text())
COLS = ["id", "lpa_app_no", "status", "decision", "decision_date", "description", "appeal_decision", "bo_system"]
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": COLS, "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]
raw = pd.DataFrame(rows).reindex(columns=COLS)
for c in COLS[1:]:
    raw[c] = raw[c].map(lambda v: v.strip() or None if isinstance(v, str) else None)

keep = pd.Series(True, index=raw.index)
for rule in rules["all_records"]:
    if rule["action"] == "drop_records":
        hit = raw["id"].isin(rule.get("ids", []))
        if "when_all_null" in rule:
            hit |= raw[rule["when_all_null"]].isna().all(axis=1)
        keep &= ~hit
c = raw[keep].copy()
portal = next(r for r in rules["all_records"] if r["id"] == "outcome_from_portal")
fill = c["id"].isin(portal["outcomes"]) & c["status"].isna() & c["decision"].isna()
c.loc[fill, ["status", "decision"]] = c.loc[fill, "id"].map(portal["outcomes"]).to_frame().values.repeat(2, axis=1)
for rule in rules["fields"]["status"]:
    if rule["action"] == "map":
        c.loc[c["status"] == rule["from"], "status"] = rule["to"]
    elif rule["action"] == "copy_when_null":
        hit = c["status"].isna() & c["decision"].notna()
        c.loc[hit, "status"] = c.loc[hit, "decision"]
    elif rule["action"] == "fill_null":
        c.loc[c["status"].isna(), "status"] = rule["to"]
for rule in rules["fields"]["decision"]:
    c.loc[c["decision"] == rule["from"], "decision"] = rule["to"]

print("Records after cleaning:", len(c))
print("decision:", c["decision"].fillna("(none)").value_counts().to_dict())
nod = c[c["decision"].isna()]
print("No decision, by status:", nod["status"].value_counts().to_dict())
legacy = nod[(nod["bo_system"] == "FME Legacy Data Import") & nod["status"].isin(["Superseded", "Lapsed"]) & nod["decision_date"].notna()]
print("Legacy Superseded/Lapsed with a decision_date, no decision:", len(legacy))
print("status Split Decision:", int((c["status"] == "Split Decision").sum()),
      "| status Withdrawn with decision Refused/Approved:", int(((c["status"] == "Withdrawn") & c["decision"].isin(["Refused", "Approved"])).sum()),
      "| status Allowed/Dismissed with decision Approved:", int((c["status"].isin(["Allowed", "Dismissed"]) & (c["decision"] == "Approved")).sum()))
print("PRE/ records:", int(c["lpa_app_no"].str.startswith("PRE/").sum()))
OBS = ["No Objection to Proposal (OBS only)", "Objection Raised to Proposal (OBS only)", "Comment Issued", "Closed"]
op = c[c["decision"].isin(OBS)]
print("Opinion-type decisions:", op["decision"].value_counts().to_dict(),
      "| of them DC/:", int(op["lpa_app_no"].str.startswith("DC").sum()))

Records after cleaning: 27765
decision: {'Approved': 20731, 'Refused': 4421, 'Withdrawn': 938, 'Not Required': 730, '(none)': 529, 'No Objection to Proposal (OBS only)': 216, 'Closed': 106, 'Comment Issued': 65, 'Objection Raised to Proposal (OBS only)': 29}
No decision, by status: {'Application Under Consideration': 481, 'Superseded': 23, 'Lapsed': 15, 'Application Invalid': 7, 'Unknown': 3}
Legacy Superseded/Lapsed with a decision_date, no decision: 38
status Split Decision: 11 | status Withdrawn with decision Refused/Approved: 16 | status Allowed/Dismissed with decision Approved: 5
PRE/ records: 37
Opinion-type decisions: {'No Objection to Proposal (OBS only)': 216, 'Closed': 106, 'Comment Issued': 65, 'Objection Raised to Proposal (OBS only)': 29} | of them DC/: 381


### Findings · Step 11

27,765 records after cleaning:

| decision | Count |
|---|---|
| Approved | 20,731 |
| Refused | 4,421 |
| Withdrawn | 938 |
| Not Required | 730 |
| (none) | **529** |
| No Objection to Proposal (OBS only) | 216 |
| Closed | 106 |
| Comment Issued | 65 |
| Objection Raised to Proposal (OBS only) | 29 |

- **No decision: 529** (not 600): Application Under Consideration 481, Superseded 23,
  Lapsed 15, Application Invalid 7, Unknown 3. The 38 legacy Superseded/Lapsed records are
  unchanged.
- Unchanged: status Split Decision 11, withdrawn appeals 16, appeals after an approval 5.
- **PRE/ records: 37** after cleaning (41 before).
- Opinion-type decisions: 416, of which 381 on DC/ records.